In [6]:
# ==============================================================================
# 1. DEPENDENCIES & SETUP
# ==============================================================================
import numpy as np
import pandas as pd
from sklearn.model_selection import LeaveOneOut, KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# Models
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import GradientBoostingRegressor, StackingRegressor

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ==============================================================================
# 2. DATA GENERATION (Simulated PCB Substrate Warpage)
# ==============================================================================
def create_pcb_dataset(n_samples=40):
    """Simulates multi-layer PCB features and non-linear warpage."""
    data = {
        'thickness_mm': np.random.uniform(0.8, 2.4, n_samples),
        'copper_density_pct': np.random.uniform(35.0, 85.0, n_samples),
        'tg_temp_C': np.random.uniform(130.0, 180.0, n_samples),
        'peak_reflow_C': np.random.uniform(235.0, 260.0, n_samples),
        'resin_pct': np.random.uniform(40.0, 60.0, n_samples),
        'unrelated_humidity': np.random.uniform(30, 60, n_samples) # Noise feature
    }
    df = pd.DataFrame(data)
    
    # Severe thermal stress when reflow temperature exceeds substrate Tg
    thermal_delta = np.maximum(0, df['peak_reflow_C'] - df['tg_temp_C'])
    
    df['warpage_microns'] = (
        (df['copper_density_pct'] * 1.5) + 
        (thermal_delta ** 1.6) * 2.0 - 
        (df['thickness_mm'] * 30.0) + 
        np.random.normal(0, 2.5, n_samples)
    )
    return df

df = create_pcb_dataset(40)
X = df.drop(columns=['warpage_microns'])
y = df['warpage_microns']

In [7]:
# ==============================================================================
# 3. DEFINE INDIVIDUAL OPTIMISED PIPELINES
# ==============================================================================

# Ridge requires scaled inputs due to L2 penalization
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=f_regression, k=4)),
    ('model', Ridge(alpha=1.0, random_state=RANDOM_STATE))
])

# Trees handle unscaled data natively but share the feature selector
gbr_pipeline = Pipeline([
    ('scaler', StandardScaler()), 
    ('selector', SelectKBest(score_func=f_regression, k=4)),
    ('model', GradientBoostingRegressor(
        n_estimators=80, 
        learning_rate=0.05, 
        max_depth=4, 
        subsample=0.9, 
        random_state=RANDOM_STATE
    ))
])

# ==============================================================================
# 4. DEFINE ENSEMBLE STACKING PIPELINE
# ==============================================================================
# Stacking couples both pipelines together, blending via a meta-learner
stack_ensemble = StackingRegressor(
    estimators=[
        ('ridge_branch', ridge_pipeline),
        ('gbr_branch', gbr_pipeline)
    ],
    final_estimator=LinearRegression(), # Meta-learner blends out-of-fold outputs [2]
    cv=5, # Internal 5-fold split to prevent data leakage during blending [2]
    n_jobs=-1
)


In [8]:
# ==============================================================================
# 5. LOOCV BENCHMARK LOOP
# ==============================================================================
loo = LeaveOneOut()

print("Evaluating Ridge Regression under LOOCV...")
y_pred_ridge = cross_val_predict(ridge_pipeline, X, y, cv=loo, n_jobs=-1)

print("Evaluating Gradient Boosting under LOOCV...")
y_pred_gbr = cross_val_predict(gbr_pipeline, X, y, cv=loo, n_jobs=-1)

print("Evaluating Stacking Ensemble under LOOCV...")
y_pred_stack = cross_val_predict(stack_ensemble, X, y, cv=loo, n_jobs=-1)


Evaluating Ridge Regression under LOOCV...
Evaluating Gradient Boosting under LOOCV...
Evaluating Stacking Ensemble under LOOCV...


In [9]:
# ==============================================================================
# 6. INDUSTRIAL EVALUATION & REPORTING
# ==============================================================================
def calculate_metrics(y_true, y_pred, model_name):
    return {
        'Model': model_name,
        'RMSE (Microns)': root_mean_squared_error(y_true, y_pred),
        'MAE (Microns)': mean_absolute_error(y_true, y_pred),
        'R² Score': r2_score(y_true, y_pred)
    }

results = [
    calculate_metrics(y, y_pred_ridge, 'Ridge Regression (Linear baseline)'),
    calculate_metrics(y, y_pred_gbr, 'Gradient Boosting (Non-linear base)'),
    calculate_metrics(y, y_pred_stack, 'Stacking Ensemble (Blended Stack)')
]

df_metrics = pd.DataFrame(results).set_index('Model')

print("\n" + "="*60)
print("             PCB SUBSTRATE EXPANSION PERFORMANCE AUDIT")
print("="*60)
print(df_metrics.round(4).to_string())
print("="*60)



             PCB SUBSTRATE EXPANSION PERFORMANCE AUDIT
                                     RMSE (Microns)  MAE (Microns)  R² Score
Model                                                                       
Ridge Regression (Linear baseline)          66.3874        48.8702    0.9926
Gradient Boosting (Non-linear base)        185.5610       155.2738    0.9424
Stacking Ensemble (Blended Stack)           69.3803        54.5812    0.9920


In [10]:
# ==============================================================================
# 7. METRIC INSPECTION & SERIALIZATION
# ==============================================================================
# Fit the stack on 100% of available data for factory deployment
stack_ensemble.fit(X, y)

meta_model = stack_ensemble.final_estimator_
ridge_weight = meta_model.coef_[0]
gbr_weight = meta_model.coef_[1]

print("\n[Meta-Learner Operational Strategy]")
print(f" -> Baseline Intercept Shift: {meta_model.intercept_:.3f} Microns")
print(f" -> Ridge (Linear) Prediction Weight: {ridge_weight:.4f}")
print(f" -> Gradient Boosting Prediction Weight: {gbr_weight:.4f}")

if gbr_weight > ridge_weight:
    print("\nConclusion: The physical system relies heavily on Gradient Boosting.")
    print("This indicates highly non-linear substrate warping properties near Tg.")
else:
    print("\nConclusion: The physical system relies heavily on Ridge.")
    print("The thermal expansion properties remain largely stable and linear.")



[Meta-Learner Operational Strategy]
 -> Baseline Intercept Shift: -10.789 Microns
 -> Ridge (Linear) Prediction Weight: 1.1411
 -> Gradient Boosting Prediction Weight: -0.1333

Conclusion: The physical system relies heavily on Ridge.
The thermal expansion properties remain largely stable and linear.
